# Feasibility Validation: Can This Schedule Actually Be Executed?

This notebook demonstrates how to **systematically validate schedule feasibility** by checking all constraints.

Understanding feasibility validation is important because:
- Infeasible schedules cannot be executed, regardless of how optimal they appear
- Constraint violations must be identified before implementation
- Different types of constraints require different checks
- Systematic validation prevents costly mistakes


## Key Concepts

**Feasibility** means the schedule can be executed:
- All hard constraints are respected
- Dependencies are not violated
- Resource capabilities match requirements
- Availability constraints are met

**Constraint Types to Check**:
1. **Time constraints**: Deadlines, time windows, durations
2. **Availability constraints**: When resources can work
3. **Capability constraints**: Skills and certifications required
4. **Fatigue constraints**: Rest periods, maximum hours, consecutive days

**Systematic Validation**:
- Check each constraint type systematically
- Identify all violations
- Determine why schedule is infeasible
- Fix violations or request new schedule

**Critical insight**: Feasibility must be validated first. An infeasible schedule cannot be "fixed" by optimization - it must be corrected.


## Scenario: Validating a Staffing Schedule

You receive a staffing schedule recommendation. Before implementing it, you must validate that it's feasible.


## Step 1: Install Required Packages (Colab)


In [1]:
%pip install pandas numpy -q



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Libraries

We use Python libraries to store validation checklists and constraints.

**What this code does:** Loads pandas (for data tables) and numpy (for calculations).

**What to look for in the output:** No output is expected here. These imports happen silently. If you see an ImportError, one of the packages may not be installed — re-run the install cell above.

In [2]:
import pandas as pd
import numpy as np


## Step 3: Systematic Feasibility Checks

This section presents a comprehensive checklist for validating whether a schedule is feasible (can actually be executed).

**What this code does:** Displays a checklist organized by constraint type: time constraints (deadlines, time windows, dependencies), availability constraints (people/resources available when scheduled), capability constraints (required skills present), and fatigue constraints (maximum hours, rest periods, consecutive days). This is a framework you can apply to any schedule.

**What to look for in the output:** This checklist is your validation tool. Use it systematically on any schedule you receive. Go through each constraint type, mark each check, and identify any violations. Even one violation makes the schedule infeasible — it cannot be executed as written and must be corrected.

In [3]:
# Example validation checklist
validation_checklist = {
    'Time Constraints': ['Deadlines met?', 'Time windows respected?', 'Dependencies not violated?'],
    'Availability Constraints': ['People scheduled when available?', 'Resources available at assigned times?'],
    'Capability Constraints': ['Required skills present?', 'Certifications valid?'],
    'Fatigue Constraints': ['Maximum hours respected?', 'Rest periods adequate?', 'Consecutive days within limit?']
}

print("FEASIBILITY VALIDATION CHECKLIST:")
print("=" * 70)
for constraint_type, checks in validation_checklist.items():
    print(f"\n{constraint_type}:")
    for check in checks:
        print(f"  ☐ {check}")

print("\nKey Insight:")
print("  - Check each constraint type systematically")
print("  - Any violation makes the schedule INFEASIBLE")
print("  - Infeasible schedules cannot be executed, regardless of optimality")
print("  - Must fix violations or request new schedule")


FEASIBILITY VALIDATION CHECKLIST:

Time Constraints:
  ☐ Deadlines met?
  ☐ Time windows respected?
  ☐ Dependencies not violated?

Availability Constraints:
  ☐ People scheduled when available?
  ☐ Resources available at assigned times?

Capability Constraints:
  ☐ Required skills present?
  ☐ Certifications valid?

Fatigue Constraints:
  ☐ Maximum hours respected?
  ☐ Rest periods adequate?
  ☐ Consecutive days within limit?

Key Insight:
  - Check each constraint type systematically
  - Any violation makes the schedule INFEASIBLE
  - Infeasible schedules cannot be executed, regardless of optimality
  - Must fix violations or request new schedule


## Summary: Feasibility Validation

**Feasibility** means the schedule can be executed:
- All hard constraints must be respected
- Any violation makes the schedule infeasible

**Systematic Validation**:
- Check time constraints (deadlines, dependencies)
- Check availability constraints
- Check capability constraints
- Check fatigue constraints

**Practical Implications**:
- Always validate feasibility before evaluating optimality
- Infeasible schedules cannot be "fixed" by optimization
- Identify all violations to understand why schedule is infeasible
- Fix violations or request new schedule with corrected constraints


## Appendix (optional): Enterprise solver view (PuLP)

The main lesson above is intentionally separate from solver mechanics. In practice, teams often encode the **same relationships** (constraints and objectives) in a modeling language and hand them to an **optimization engine**.

- **Python modeling:** [PuLP](https://coin-or.github.io/pulp/) is widely used in teaching and analytics; it ships with the open-source **CBC** solver.
- **Commercial solvers:** **Gurobi**, **IBM CPLEX**, and **FICO Xpress** are common in enterprises (similar algebraic models, different backends).
- **Another popular stack:** [Google OR-Tools](https://developers.google.com/optimization) (CP-SAT, routing, scheduling).

Install when needed (Colab or local):

```python
%pip install pulp -q
```

The code below is a compact **parallel** to the scenario in this notebook—not a replacement for the conceptual steps above.

**This appendix:** when constraints contradict each other, a solver returns **Infeasible**—the same conclusion as a manual checklist, with automation.


In [4]:
%pip install pulp -q

from pulp import LpMinimize, LpProblem, LpStatus, LpVariable, PULP_CBC_CMD, value

# Tiny precedence model that contradicts a tight deadline -> solver returns Infeasible
m = LpProblem("impossible_deadline", LpMinimize)
s_d = LpVariable("start_design", lowBound=0)
s_dev = LpVariable("start_dev", lowBound=0)
C = LpVariable("finish", lowBound=0)
m += C
m += s_dev >= s_d + 5   # design 5d before dev
m += C >= s_dev + 10    # dev 10d
m += C <= 12            # deadline infeasible with chain 5+10

m.solve(PULP_CBC_CMD(msg=False))
print("Solver status:", LpStatus[m.status])
print("In practice, validation and modeling tools surface this before anyone publishes a Gantt chart.")



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Solver status: Infeasible
In practice, validation and modeling tools surface this before anyone publishes a Gantt chart.
